# Primeiras Análises do conjunto de dados:

In [38]:
import pandas as pd
import plotly.express as px
import streamlit as st

# subir um nível (..) para achar a pasta 'data'
imdb = pd.read_csv('../data/world_imdb_movies_top_movies_per_year.csv')

print("Dados carregados com sucesso!", imdb.info())
print()
print(imdb.columns)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33600 entries, 0 to 33599
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     33600 non-null  object 
 1   title                  33600 non-null  object 
 2   link                   33600 non-null  object 
 3   year                   33600 non-null  int64  
 4   duration               33379 non-null  object 
 5   rating_mpa             25624 non-null  object 
 6   rating_imdb            33462 non-null  float64
 7   vote                   33462 non-null  float64
 8   budget                 11815 non-null  float64
 9   gross_world_wide       18222 non-null  float64
 10  gross_us_canada        17571 non-null  float64
 11  gross_opening_weekend  15523 non-null  float64
 12  director               33241 non-null  object 
 13  writer                 32024 non-null  object 
 14  star                   33127 non-null  object 
 15  ge

In [13]:
# exibição de uma amostra dos dados para verificar se estão corretos
display(imdb.sample(5))

#verificar se há valores nulos
print(imdb.isnull().sum())

,id,title,link,year,duration,rating_mpa,rating_imdb,vote,budget,gross_world_wide,...,writer,star,genre,country_origin,filming_location,production_company,language,win,nomination,oscar
21181,tt0099920,Kaninmannen,https://www.imdb.com/title/tt0099920,1990,1h 37m,NaN,5.6,220.0,NaN,NaN,...,Stig Larsson,"Brje Ahlstedt, Leif Andre, Stina Ekblad",Drama,Sweden,"Kvarnbackaskolan, Kista, Stockolm, Sweden","Omega Film Television, Svenska Filminstitutet...",Swedish,0,0,0
16247,tt0491162,Spiral,https://www.imdb.com/title/tt0491162,2007,1h 30m,PG-13,6.3,4000.0,600000.0,3072.0,...,"Jeremy Boreing, Joel David Moore","Joel David Moore, Amber Tamblyn, Zachary Levi","Drama, Thriller",United States,"136 NW 9th Avenue, Portland, Oregon, USA","ArieScope Pictures, Coattails Entertainment",English,0,0,0
13349,tt0054326,Sons and Lovers,https://www.imdb.com/title/tt0054326,1960,1h 43m,Approved,7.1,18000.0,500000.0,NaN,...,"Gavin Lambert, TEB Clarke, DH Lawrence","Trevor Howard, Dean Stockwell, Wendy Hiller",Drama,United Kingdom,"Nottingham, Nottinghamshire, England, UK",Jerry Wald Productions,English,0,16,0
26079,tt0379445,Prem Parvat,https://www.imdb.com/title/tt0379445,1973,NaN,NaN,6.9,17.0,NaN,NaN,...,NaN,"Birbal, Hiralal, Satish Kaul","Drama, Romance",India,NaN,NaN,Hindi,0,0,0
16552,tt1212451,April Showers,https://www.imdb.com/title/tt1212451,2009,1h 34m,R,5.6,1000.0,1100000.0,16880.0,...,Andrew Robinson,"Kelly Blatz, Daryl Sabara, Janel Parrish","True Crime, Crime, Drama",United States,"Omaha, Nebraska, USA",April Showers,English,0,0,0


id                           0
title                        0
link                         0
year                         0
duration                   221
rating_mpa                7976
rating_imdb                138
vote                       138
budget                   21785
gross_world_wide         15378
gross_us_canada          16029
gross_opening_weekend    18077
director                   359
writer                    1576
star                       473
genre                      382
country_origin             366
filming_location          6729
production_company        1378
language                   491
win                          0
nomination                   0
oscar                        0
dtype: int64


In [14]:
# Limpeza geral dos dados

# Remover linhas onde o ano ou título estão faltando, porque não fariam sentido para a nossa análise
imdb = imdb.dropna(subset=['year', 'title'])

# Finanças sem valores vazios nestas colunas, posso criar um novo dataframe apenas para analisar as finanças
df_financeiro = imdb.dropna(subset=['budget', 'gross_world_wide'])

# Criar uma coluna apenas com o gênero principal do filme, para facilitar a análise
imdb['genre_principal'] = imdb['genre'].str.split(',').str[0]

# Transformar NaN em "Não Classificado" na coluna que indica a classificação indicativa do filme
imdb['rating_mpa'] = imdb['rating_mpa'].fillna('Not Rated')

In [16]:
# Lista das colunas que são mais relevantes para a análise exploratória
colunas_selecionadas = [
    'title', 'year', 'duration', 'rating_mpa', 'rating_imdb', 
    'vote', 'genre', 'genre_principal', 'director', 'oscar'
]

# Aplicando o filtro (Lembrando que df_financeiro já salvou o que era dinheiro antes)
imdb = imdb[colunas_selecionadas]

imdb.sample(5)

,title,year,duration,rating_mpa,rating_imdb,vote,genre,genre_principal,director,oscar
3634,Tower House,1962,2h 15m,Not Rated,5.1,43.0,"Action, Drama, Horror, Musical, Thriller",Action,Nisar Ahmad Ansari,0
27003,Nyi Ageng Ratu Pemikat,1983,1h 23m,Not Rated,7.8,164.0,Fantasy,Fantasy,Sisworo Gautama Putra,0
13397,Naaraaz,1994,2h 8m,Not Rated,4.7,192.0,"Action, Crime, Drama",Action,Mahesh Bhatt,0
23595,Save Yourselves!,2020,1h 33m,R,5.8,88000.0,"Adventure, Comedy, SciFi",Adventure,"Alex Huston Fischer, Eleanor Wilson",0
13314,Heat,1972,1h 42m,R,6.1,19000.0,"Satire, Comedy, Drama, Romance",Satire,Paul Morrissey,0


# Usarei agora a Plotagem de gráficos com os dados presentes para ir mais a fundo na análise

In [21]:
# 1. Preparamos os dados: contamos e pegamos apenas os 15 mais frequentes
top_generos = imdb['genre_principal'].value_counts().nlargest(15).reset_index()

# 2. Criamos o gráfico horizontal (trocamos x por y)
fig1 = px.bar(top_generos, 
            y='genre_principal', # Gênero no eixo Y (horizontal)
            x='count',           # Quantidade no eixo X
            title="Top 15 Gêneros Mais Frequentes no IMDB",
            labels={'count': 'Número de Filmes', 'genre_principal': 'Gênero'},
            color='count',       # Cor baseada na quantidade
            color_continuous_scale='Viridis', # Escala de cor profissional
            orientation='h')     # Força a orientação horizontal

# 3. Melhoramos o layout (ajuste de margens e ordem)
fig1.update_layout(yaxis={'categoryorder':'total ascending'}, # Ordena do maior para o menor
                height=600) # Aumenta a altura para os nomes não ficarem apertados

fig1.show()

In [35]:
# usando o df_financeiro que tem os valores de dinheiro)
# agrupar por ter ou não Oscar e calcular a média da bilheteria
df_oscar_money = df_financeiro.groupby('oscar')['gross_world_wide'].mean().reset_index()

# o gráfico de barras
fig_oscar_bar = px.bar(df_oscar_money, 
                    x='oscar', 
                    y='gross_world_wide',
                    color='oscar',
                    title="Média de Faturamento Mundial: Com Oscar vs. Sem Oscar",
                    labels={'gross_world_wide': 'Faturamento Médio ($)', 'oscar': 'Ganhou Oscar?'},
                    text_auto='.2s') # Mostra o valor abreviado em cima da barra (ex: 150M)

fig_oscar_bar.show()

In [34]:
# O "Oscar" traz nota ou traz dinheiro?
# Filtrar apenas quem tem pelo menos 1 Oscar
vencedores = df_financeiro[df_financeiro['oscar'] > 0]

fig_vencedores = px.scatter(vencedores, 
                            x="rating_imdb", 
                            y="gross_world_wide",
                            size="oscar", # Bolas maiores para quem tem MAIS Oscars
                            color="year",
                            hover_name="title",
                            title="Vencedores de Oscar: Nota vs. Bilheteria ao longo dos anos")

fig_vencedores.show()

In [36]:
# Analise de evolução da nota máxima do IMDB por ano
# Agrupar por ano e pegar a nota máxima de cada um
notas_por_ano = imdb.groupby('year')['rating_imdb'].max().reset_index()

# Criar o gráfico de linha
fig_linha = px.line(notas_por_ano, 
                    x="year", 
                    y="rating_imdb",
                    title="Evolução da Maior Nota do IMDB por Ano",
                    markers=True,
                    labels={'rating_imdb': 'Nota Máxima', 'year': 'Ano'})

# (focar entre as notas 7 e 10)
fig_linha.update_yaxes(range=[7, 10])

fig_linha.show()

In [41]:
# prototipo do gráfico de barras para o Oscar (sem as cores e ajustes finais)
# adição ao streamlit depois, mas aqui é para testar o gráfico mesmo
fig_oscar = px.bar(df_oscar_money,
                    x='oscar',
                    y='gross_world_wide',
                    color='oscar',
                    title="Média de Faturamento Mundial: Com Oscar vs. Sem Oscar",
                    labels={
                        'gross_world_wide': 'Faturamento Médio ($)', 'oscar': 'Ganhou Oscar?'},
                    text_auto=True,
                    # Prata e Ouro!
                    color_discrete_map={'Não': '#C0C0C0', 'Sim': '#FFD700'})
fig_oscar.show()

# st.plotly_chart(fig_oscar, use_container_width=True)